In [1]:
"""Task A: Data Pipeline for Time Series Forecasting"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import os

class WindowedTimeSeries(Dataset):
    """Sliding window dataset for time series forecasting"""
    
    def __init__(self, data, targets, lookback=96, horizon=24):
        """
        Args:
            data: Input features (n_samples, n_features)
            targets: Target values (n_samples, n_targets)
            lookback: Number of past steps to use (L)
            horizon: Number of future steps to predict (H)
        """
        self.data = data
        self.targets = targets
        self.L = lookback
        self.H = horizon
        
    def __len__(self):
        return len(self.data) - self.L - self.H + 1
    
    def __getitem__(self, idx):
        # Input: past L steps
        x = self.data[idx:idx + self.L]  # (L, d)
        
        # Output: future H steps
        y = self.targets[idx + self.L:idx + self.L + self.H]  # (H, out_dim)
        
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )

def load_ett_data(data_path=None, target_col="OT"):
    """
    Load ETTh1 dataset
    ETTh1: Electricity Transformer Temperature - hourly data
    """
    if data_path is None:
        # Auto-download if not exists
        url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
        df = pd.read_csv(url)
    else:
        df = pd.read_csv(data_path)
    
    # Parse date column
    df['date'] = pd.to_datetime(df['date'])
    
    # Drop date column for modeling
    dates = df['date']
    data = df.drop(columns=['date'])
    
    # Features: 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT'
    feature_names = data.columns.tolist()
    print(f"Features: {feature_names}")
    print(f"Data shape: {data.shape}")
    
    return data.values, dates, feature_names

def prepare_data(data, target_idx=0, train_ratio=0.7, val_ratio=0.1, test_ratio=0.2):
    """
    Split data into train/val/test and normalize
    
    Args:
        data: numpy array of shape (n_samples, n_features)
        target_idx: Index of target variable (0 for OT)
        train_ratio: Training set proportion
        val_ratio: Validation set proportion
        test_ratio: Test set proportion
    
    Returns:
        train_loader, val_loader, test_loader, scaler
    """
    n = len(data)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_data = data[:train_end]
    val_data = data[train_end:val_end]
    test_data = data[val_end:]
    
    # Normalize using training statistics only
    scaler = StandardScaler()
    train_data_norm = scaler.fit_transform(train_data)
    val_data_norm = scaler.transform(val_data)
    test_data_norm = scaler.transform(test_data)
    
    # Extract target (single variable forecasting)
    train_target = train_data_norm[:, target_idx:target_idx+1]
    val_target = val_data_norm[:, target_idx:target_idx+1]
    test_target = test_data_norm[:, target_idx:target_idx+1]
    
    # Features (all variables including target for input)
    train_features = train_data_norm
    val_features = val_data_norm
    test_features = test_data_norm
    
    # Create datasets
    lookback = 96
    horizon = 24
    
    train_dataset = WindowedTimeSeries(train_features, train_target, lookback, horizon)
    val_dataset = WindowedTimeSeries(val_features, val_target, lookback, horizon)
    test_dataset = WindowedTimeSeries(test_features, test_target, lookback, horizon)
    
    return train_dataset, val_dataset, test_dataset, scaler

def get_dataloaders(batch_size=64):
    """Get all dataloaders"""
    data, dates, features = load_ett_data()
    train_ds, val_ds, test_ds, scaler = prepare_data(data, target_idx=0)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, scaler, data.shape[1]

if __name__ == "__main__":
    # Test the data pipeline
    train_loader, val_loader, test_loader, scaler, input_dim = get_dataloaders()
    
    print(f"Input dimension: {input_dim}")
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")
    
    # Inspect a batch
    for x, y in train_loader:
        print(f"Input shape: {x.shape}")  # (B, L, d)
        print(f"Target shape: {y.shape}")  # (B, H, 1)
        break

Features: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
Data shape: (17420, 7)
Input dimension: 7
Train batches: 189
Val batches: 26
Test batches: 53
Input shape: torch.Size([64, 96, 7])
Target shape: torch.Size([64, 24, 1])
